# Web Scraper Agent — Fetch, Parse, Index, Answer
## Crawl, Parse, Index, Answer — The Full Web RAG Pipeline
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/60-web-scraper-agent/web_scraper_workbook.ipynb)

RAG doesn't have to start from a pre-built index. This workshop builds a **live web scraper RAG pipeline**: fetch a URL, strip HTML boilerplate with BeautifulSoup, chunk the text, embed it into an ephemeral ChromaDB collection, retrieve the top-k chunks, and answer a question — all in one LangGraph run. By the end you will understand *how* to build on-the-fly RAG from any public URL and *how* chunking parameters affect retrieval quality.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Live RAG vs pre-built index |
| 2 | **Fetch + Parse** — HTTP request → clean text via BeautifulSoup |
| 3 | **Chunk + Index** — RecursiveCharacterTextSplitter → ChromaDB |
| 4 | **Retrieve + Answer** — Top-k similarity search → LLM |
| 5 | **Full Pipeline** — fetch → parse → index_and_retrieve → answer |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- Internet access (fetches a Wikipedia page)

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langchain-chroma", "langchain-text-splitters",
        "chromadb", "langgraph", "beautifulsoup4", "requests", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — Live RAG vs Pre-Built Index

| | Pre-Built Index | Live Web RAG |
|---|---|---|
| When indexed | Offline batch job | At query time |
| Freshness | Stale until re-indexed | Always current |
| Latency | Fast (index already exists) | Slow (fetch + embed per query) |
| Use case | Known corpus, high-volume | Arbitrary URLs, low-volume |

**Pipeline:**
```
URL → fetch() → parse_html() → chunk_text() → embed() → Chroma → similarity_search() → LLM
```

In [ ]:
# Smaller chunks = better precision; larger chunks = more context per hit.
# CHUNK_OVERLAP prevents a sentence from being split across two chunks and lost.
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
# TOP_K=4 balances recall (more docs) vs context length (LLM token cost).
TOP_K         = 4
TARGET_URL    = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"

QUESTIONS = [
    "What is retrieval-augmented generation?",
    "Who introduced RAG and when?",
    "What are the main components of a RAG system?",
]
print(f"Target: {TARGET_URL}")
print(f"Chunk size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}, top_k={TOP_K}")

---
## Part 2 — Fetch + Parse

In [ ]:
import requests
from bs4 import BeautifulSoup

# Wikipedia blocks the default Python requests UA; a browser UA avoids 403s.
def fetch_page(url: str) -> str:
    r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    return r.text

def parse_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
# Strip navigation chrome first — it adds tokens without answering questions.
    for tag in soup(["nav", "footer", "script", "style", "header"]):
# Prefer semantic tags (main, article) over body — less boilerplate text.
        tag.decompose()
    main = soup.find("main") or soup.find("article") or soup.find("body")
    return main.get_text(separator=" ", strip=True) if main else soup.get_text(separator=" ", strip=True)

html = fetch_page(TARGET_URL)
text = parse_html(html)
print(f"Fetched {len(html):,} bytes of HTML → {len(text):,} chars of clean text")
print(f"Preview: {text[:300]}...")

---
## Part 3 — Chunk + Index

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter tries '\n\n', '\n', ' ' in order — keeps paragraphs intact when possible.
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_text(text)

# text-embedding-3-small is 5x cheaper than ada-002 with similar retrieval quality.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# Chroma is in-process — no server needed; swap for Pinecone/Weaviate for production scale.
vectorstore = Chroma.from_texts(chunks, embeddings, collection_name="web-scraper")

print(f"Split into {len(chunks)} chunks  |  Index built: {len(chunks)} vectors")
print(f"\nSample chunk [0]: {chunks[0][:200]}")

---
## Part 4 — Retrieve + Answer

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# temperature=0 forces deterministic answers — avoids hallucination drift across runs.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

for q in QUESTIONS:
# similarity_search returns the k nearest vectors by cosine distance — k=TOP_K set above.
    docs = vectorstore.similarity_search(q, k=TOP_K)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"Use this context to answer the question.\n\nContext:\n{context}\n\nQuestion: {q}"
    answer = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"  (used {len(docs)} chunks, {sum(len(d.page_content) for d in docs)} chars of context)")
    print()

---
## Part 5 — Full LangGraph Pipeline

```
START → fetch → parse → index_and_retrieve → answer → END
```

In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

# TypedDict gives LangGraph typed access to state keys; every node reads/writes this dict.
class WebScraperState(TypedDict):
    url: str; question: str; raw_html: str
    text: str; chunks: list; context: list; answer: str

def fetch_node(state):
    return {"raw_html": fetch_page(state["url"])}

def parse_node(state):
    t = parse_html(state["raw_html"])
    return {"text": t, "chunks": splitter.split_text(t)}

def index_retrieve_node(state):
    vs = Chroma.from_texts(state["chunks"], embeddings, collection_name="ws-run")
    docs = vs.similarity_search(state["question"], k=TOP_K)
    return {"context": [d.page_content for d in docs]}

def answer_node(state):
    ctx = "\n\n".join(state["context"])
    prompt = f"Use this context to answer the question.\n\nContext:\n{ctx}\n\nQuestion: {state['question']}"
    return {"answer": llm.invoke([HumanMessage(content=prompt)]).content.strip()}

# compile() locks the graph topology — no nodes or edges can be added after this point.
g = StateGraph(WebScraperState)
for name, fn in [("fetch", fetch_node), ("parse", parse_node),
                 ("index_and_retrieve", index_retrieve_node), ("answer", answer_node)]:
    g.add_node(name, fn)
g.add_edge(START, "fetch"); g.add_edge("fetch", "parse")
g.add_edge("parse", "index_and_retrieve"); g.add_edge("index_and_retrieve", "answer")
g.add_edge("answer", END)
# add_edge wires nodes in sequence; add_conditional_edges would branch on node return value.
app = g.compile()

result = app.invoke({"url": TARGET_URL, "question": QUESTIONS[0],
                     "raw_html": "", "text": "", "chunks": [], "context": [], "answer": ""})
print(f"Q: {result['question']}")
print(f"Chunks: {len(result['chunks'])}  |  Context docs: {len(result['context'])}")
print(f"A: {result['answer']}")

---
### Exercise 1 — Change the URL
Replace `TARGET_URL` with another Wikipedia article (e.g., `https://en.wikipedia.org/wiki/LangChain`) and ask a question about it. Does the pipeline work without any code changes?

### Exercise 2 — Tune Chunk Size
Try `CHUNK_SIZE=200` and `CHUNK_SIZE=1000`. How does the number of chunks change? Do smaller or larger chunks produce better answers?

In [ ]:
# Exercise 1 — swap URL
NEW_URL = "https://en.wikipedia.org/wiki/LangChain"  # replace as desired
NEW_Q   = "What is LangChain used for?"
# TODO: invoke app with NEW_URL and NEW_Q

# Exercise 2 — chunk size sweep
for cs in [200, 500, 1000]:
    sp = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=50)
    ch = sp.split_text(text)
    print(f"chunk_size={cs:4d} → {len(ch)} chunks, avg {sum(len(c) for c in ch)//len(ch)} chars/chunk")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*